## Conversational memory

So far every call to `rag_chain_lcel` is **stateless**: the chain forgets the question as soon as it answers. That's fine for one-off questions, but it breaks follow-ups:

```
You: What is reinforcement learning?
Bot: Reinforcement learning is ... trial and error ... rewards ...

You: What are its main challenges?
         │
         ▼
 retriever searches for "What are its main challenges?"
 → "its" could mean anything, so it finds the wrong chunks
 → the LLM never saw the first question, so it doesn't know what "its" means
```

**Conversational memory** fixes this by keeping the **chat history** (the past questions and answers) and giving it to the chain on every new turn.

> **In one sentence:** memory = save the conversation, and send it back to the model with each new question.

### Two jobs in a conversational RAG chain

```
question + chat_history
        │
        ▼
 1. Rewrite the question       LLM: "What are its main challenges?"
        │                        → "What are the main challenges of reinforcement learning?"
        ▼
 2. Retrieve                   retriever searches with the standalone question
        │
        ▼
 3. Answer                     prompt = context + chat_history + question → LLM → answer
        │
        ▼
 4. Save the turn              chat_history += [question, answer]
```

1. **Rewrite the follow-up into a standalone question.** The retriever only sees one string, so it can't use the history. An LLM first rewrites the question so it makes sense on its own. This is also called *query contextualization*, or a *history-aware retriever*.
2. **Give the history to the answer prompt.** `MessagesPlaceholder("chat_history")` adds the past messages to the prompt, so the LLM can answer in context.

### Sample code with LCEL

```python
from operator import itemgetter
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough

# 1. Rewrite a follow-up question into a standalone question
contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Given the chat history and the latest user question, rewrite the question so it "
     "can be understood without the chat history. Do NOT answer it, only rewrite it. "
     "If it is already a standalone question, return it unchanged."),
    MessagesPlaceholder("chat_history"),
    ("human", "{question}"),
])
contextualize_chain = contextualize_prompt | llm | StrOutputParser()

# 2. Answer using the retrieved context AND the chat history
qa_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer using only the context below. If the answer is not in the context, "
     "say you don't know.\n\nContext:\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{question}"),
])

conversational_rag_chain = (
    RunnablePassthrough.assign(standalone_question=contextualize_chain)
    | RunnablePassthrough.assign(
        context=itemgetter("standalone_question") | retriever | format_docs
    )
    | qa_prompt
    | llm
    | StrOutputParser()
)

# 3. Keep the history in a list and add every turn to it
chat_history = []

def ask_with_memory(question):
    answer = conversational_rag_chain.invoke(
        {"question": question, "chat_history": chat_history}
    )
    chat_history.extend([HumanMessage(content=question), AIMessage(content=answer)])
    print(f"Question: {question}\n\nAnswer:\n{answer}\n")
    return answer

ask_with_memory("What is reinforcement learning?")
ask_with_memory("What are its main challenges?")   # "its" now means reinforcement learning
```

How the chain works:

- `RunnablePassthrough.assign(...)` keeps the input dict (`question`, `chat_history`) and **adds a new key** to it: first `standalone_question`, then `context`.
- `itemgetter("standalone_question")` picks the rewritten question out of the dict, so the retriever searches with it instead of the original follow-up.
- `chat_history.extend([...])` adds the question and the answer as two messages after each turn.

### Types of memory

The history grows with every turn, and a long history costs more tokens and can hit the model's context limit. Common ways to control it:

| Type | What is kept | Trade-off |
|---|---|---|
| **Buffer** (full history) | Every message | Simplest, but the prompt keeps growing |
| **Window** | Only the last *k* turns, e.g. `chat_history[-6:]` for the last 3 Q&A pairs | Cheap and predictable, but forgets older details |
| **Summary** | An LLM-written summary of older turns, plus the latest messages | Keeps long conversations short, but costs extra LLM calls and loses some detail |
| **Vector store memory** | Past messages embedded and searched like documents | Can recall things from long ago, but is more complex |

### Where the history lives

| Option | When to use it |
|---|---|
| A Python list (as above) | Notebooks and quick tests; lost when the kernel restarts |
| `RunnableWithMessageHistory` + `InMemoryChatMessageHistory` | One separate history per user, keyed by a `session_id` |
| A database (Redis, Postgres, SQLite, ...) | Production apps where history must survive restarts |
| LangGraph checkpointer (`InMemorySaver`, or a database saver) | Agents built with LangGraph or `create_agent`: memory is saved per `thread_id` |

`langchain_classic` also has a ready-made version of step 1, `create_history_aware_retriever(llm, retriever, prompt)`, which plugs into `create_retrieval_chain` like the chain in the *Modern RAG Chain* section.

In [1]:
## Setup: shared project paths from src.config, which also loads the API key from .env
from pathlib import Path

from src.config import PROJECT_ROOT, CHROMA_PATH

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp


In [2]:
## Load the Chroma vector store built in 01_chromadb.ipynb (no re-embedding needed)
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# Must be the same embedding model that was used to build the store
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma(
    collection_name="text_docs",
    embedding_function=embeddings,
    persist_directory=str(CHROMA_PATH),   # Chroma expects a str, not a Path
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

llm = ChatOpenAI(model="gpt-5", temperature=0)

print(f"Loaded {vectorstore._collection.count()} chunks from {CHROMA_PATH}")

/var/folders/j5/52bx342j7k59wyrt9n978pp00000gn/T/ipykernel_31796/3439046416.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma
/var/folders/j5/52bx342j7k59wyrt9n978pp00000gn/T/ipykernel_31796/3439046416.py:8: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


Loaded 41 chunks from /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/vectorstore/chroma_db


In [3]:
## Step 1: history-aware retriever (rewrites a follow-up question before searching)
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Given the chat history and the latest user question, which might refer to the "
     "chat history, rewrite it as a standalone question that can be understood without "
     "the chat history. Do NOT answer the question: only rewrite it if needed, "
     "otherwise return it as is."),
    MessagesPlaceholder("chat_history"),   # the past HumanMessage / AIMessage objects go here
    ("human", "{input}"),
])

# No chat history -> the question goes straight to the retriever
# With chat history -> prompt | llm rewrites the question first, then the retriever searches with it
history_aware_retriever = create_history_aware_retriever(llm, retriever, contextualize_q_prompt)
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x1116f67b0>, search_kwargs={'k': 3}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')] | typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')] | typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')] | typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')] | typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')] | typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')] | typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageCh

In [4]:
## Step 2: answer chain that sees the retrieved context AND the chat history
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import PromptTemplate

qa_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant. Answer using only the retrieved context below. "
     "If the answer is not in the context, say you don't know based on the provided documents. "
     "Cite sources using the [Source: ...] labels. Keep the answer concise.\n\n"
     "Context:\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

document_prompt = PromptTemplate.from_template("[Source: {source}]\n{page_content}")

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt, document_prompt=document_prompt)
conversational_rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)
conversational_rag_chain

/Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x1116f67b0>, search_kwargs={'k': 3}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')] | typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')] | typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')] | typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')] | typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')] | typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')] | typing.Annot

In [5]:
## Step 3: keep the chat history as HumanMessage / AIMessage objects
from langchain_core.messages import HumanMessage, AIMessage

chat_history = []

question_1 = "What is reinforcement learning?"
result_1 = conversational_rag_chain.invoke({"input": question_1, "chat_history": chat_history})

# Save this turn: the user's question and the model's answer
chat_history.extend([HumanMessage(content=question_1), AIMessage(content=result_1["answer"])])

print(f"Question: {question_1}\n")
print(f"Answer:\n{result_1['answer']}")

Question: What is reinforcement learning?

Answer:
Reinforcement learning is a branch of machine learning where an agent learns through trial and error by interacting with an environment, taking actions, receiving rewards or penalties, and learning a strategy (policy) that maximizes total reward over time; it’s used in areas like game playing, robotics, and fine-tuning large language models with human feedback. [Source: /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/data/raw/text/reinforcement_learning.txt] [Source: /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/data/raw/text/machine_learning_algorithms.txt]


In [6]:
## Follow-up question: "its" refers to reinforcement learning from the previous turn
question_2 = "What are its main challenges?"
result_2 = conversational_rag_chain.invoke({"input": question_2, "chat_history": chat_history})

chat_history.extend([HumanMessage(content=question_2), AIMessage(content=result_2["answer"])])

print(f"Question: {question_2}\n")
print(f"Answer:\n{result_2['answer']}\n")
print("Retrieved sources:")
for source in dict.fromkeys(doc.metadata.get("source", "unknown") for doc in result_2["context"]):
    print(f"- {source}")

Question: What are its main challenges?

Answer:
- Requires a very large number of interactions, making learning slow or expensive in real-world settings
- Hard to design good reward functions; poor rewards can cause “reward hacking”
- Training can be unstable
- Policies learned in simulation often don’t transfer well to the real world

[Source: /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/data/raw/text/reinforcement_learning.txt]

Retrieved sources:
- /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/data/raw/text/reinforcement_learning.txt


In [7]:
## Peek inside step 1: the standalone question the retriever actually searched with
from langchain_core.output_parsers import StrOutputParser

rewrite_chain = contextualize_q_prompt | llm | StrOutputParser()

# chat_history[:2] is the history as it was when question_2 was asked (the first Q&A pair)
rewrite_chain.invoke({"input": question_2, "chat_history": chat_history[:2]})

'What are the main challenges of reinforcement learning?'

In [8]:
## Helper: ask a question, answer it with the chat history, then save the turn
def chat_with_memory(question):
    result = conversational_rag_chain.invoke({"input": question, "chat_history": chat_history})
    chat_history.extend([HumanMessage(content=question), AIMessage(content=result["answer"])])
    print(f"Question: {question}\n\nAnswer:\n{result['answer']}\n")
    return result["answer"]


answer = chat_with_memory("What are the core components of an AI agent?")
answer = chat_with_memory("How does it use tools?")   # "it" = an AI agent

Question: What are the core components of an AI agent?

Answer:
- Agent: the learner/decision-maker
- Environment: the world it interacts with
- Actions: choices the agent takes
- Rewards/Penalties: feedback from the environment
- Strategy (policy): the learned way of acting to maximize total reward over time

[Source: /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/data/raw/text/machine_learning_algorithms.txt] [Source: /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/data/raw/text/reinforcement_learning.txt]

Question: How does it use tools?

Answer:
- The LLM (the agent’s “brain”) decides when to call external tools (the agent’s “hands”) like web search or a calculator.
- It plans multiple steps, invokes tools, inspects the results, and iterates until the task is finished.
- Retrieval systems like RAG serve as tools to provide accurate, up-to-date knowledge.

[Source: /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/data/raw/text/ai_agents.txt]



In [9]:
## The full chat history: one HumanMessage and one AIMessage per turn
print(f"{len(chat_history)} messages in the history\n")
for message in chat_history:
    print(f"{message.type:>5}: {message.content[:80]}...")

8 messages in the history

human: What is reinforcement learning?...
   ai: Reinforcement learning is a branch of machine learning where an agent learns thr...
human: What are its main challenges?...
   ai: - Requires a very large number of interactions, making learning slow or expensiv...
human: What are the core components of an AI agent?...
   ai: - Agent: the learner/decision-maker
- Environment: the world it interacts with
-...
human: How does it use tools?...
   ai: - The LLM (the agent’s “brain”) decides when to call external tools (the agent’s...
